In [1]:
import json

with open("sample1.json", encoding="utf-8") as f:
    meeting = json.load(f)

print(meeting["meeting_id"], "-", len(meeting["segments"]), "segments")
for seg in meeting["segments"][:3]:
    print(seg["speaker"], ":", seg["text"])

sample1 - 20 segments
Interviewer : What project did you work on?
Student : I made an English poster about our environment.
Interviewer : Was that an indoor or outdoor project?


In [2]:
from transformers import pipeline

ner = pipeline("ner", model="dslim/bert-base-NER",
               aggregation_strategy="simple", device=0)

for seg in meeting["segments"]:
    ents = ner(seg["text"])
    seg["entities"] = [
        {"text": e["word"], "type": e["entity_group"], "score": round(float(e["score"]), 2)}
        for e in ents
    ]
    if seg["entities"]:
        print(seg["speaker"], "|", seg["text"])
        for e in seg["entities"]:
            print("    ", e["type"], "-", e["text"], e["score"])

config.json:   0%|          | 0.00/829 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  433MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/59.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Student | I made an English poster about our environment.
     MISC - English 1.0
Student | Our science teacher, Mr. John, we learned a lot from him.
     PER - John 0.5
Student | and I particularly liked Michael, the leader of my team.
     PER - Michael 1.0


In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

name = "philschmid/bart-large-cnn-samsum"
tokenizer = AutoTokenizer.from_pretrained(name)
model = AutoModelForSeq2SeqLM.from_pretrained(name).to("cuda")

text = "\n".join(f'{s["speaker"]}: {s["text"]}' for s in meeting["segments"])

inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=1024).to("cuda")
ids = model.generate(**inputs, max_length=120, min_length=30,
                     num_beams=4, do_sample=False)
summary = tokenizer.decode(ids[0], skip_special_tokens=True)
print(summary)

tokenizer_config.json:   0%|          | 0.00/300 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 1.63GB            

pytorch_model.bin: downloading bytes:           |  0.00B            

[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.63GB            

model.safetensors: downloading bytes:           |  0.00B            

Student made an English poster about the environment. The project was carried out mostly outdoor, but the last phase was conducted indoor. It was a successful project and received compliments from the professor. He learned how to write contents and decorate posters.


In [5]:
from collections import defaultdict

report = defaultdict(lambda: {"turns": 0, "seconds": 0.0, "entities": set(), "lines": []})

for seg in meeting["segments"]:
    r = report[seg["speaker"]]
    r["turns"] += 1
    r["seconds"] += seg["end"] - seg["start"]
    r["lines"].append(seg["text"])
    for e in seg.get("entities", []):
        r["entities"].add(f'{e["text"]} ({e["type"]})')

print("SUMMARY:", summary, "\n")
for speaker, r in report.items():
    print(f"=== {speaker} ===")
    print(f"Segments: {r['turns']}   Talk time: {r['seconds']:.0f} s")
    print("Entities:", ", ".join(sorted(r["entities"])) or "none")
    print("Said:")
    for line in r["lines"]:
        print("  -", line)
    print()

SUMMARY: Student made an English poster about the environment. The project was carried out mostly outdoor, but the last phase was conducted indoor. It was a successful project and received compliments from the professor. He learned how to write contents and decorate posters. 

=== Interviewer ===
Segments: 9   Talk time: 24 s
Entities: none
Said:
  - What project did you work on?
  - Was that an indoor or outdoor project?
  - Who was involved in the project?
  - What did you do?
  - Was it a successful project?
  - Who was your advisory teacher?
  - What did you learn after the project?
  - Did you like your partners?
  - What did you get after the project?

=== Student ===
Segments: 11   Talk time: 40 s
Entities: English (MISC), John (PER), Michael (PER)
Said:
  - I made an English poster about our environment.
  - The project was carried out mostly outdoor.
  - Only the last phase was conducted indoor.
  - My teammates and an advisory teacher.
  - We had a field trip to the countrysi

In [6]:
summary_single = summary   # keep the first summary for comparison

def summarize(txt, max_len=80, min_len=15):
    inputs = tokenizer(txt, return_tensors="pt", truncation=True, max_length=1024).to("cuda")
    ids = model.generate(**inputs, max_length=max_len, min_length=min_len,
                         num_beams=4, no_repeat_ngram_size=3, do_sample=False)
    return tokenizer.decode(ids[0], skip_special_tokens=True)

# split into parts: start a new part before a question, once the part has 5+ lines
segs = meeting["segments"]
chunks, cur = [], []
for s in segs:
    if len(cur) >= 5 and s["text"].strip().endswith("?"):
        chunks.append(cur)
        cur = []
    cur.append(s)
chunks.append(cur)

part_summaries = []
for i, chunk in enumerate(chunks, 1):
    chunk_text = "\n".join(f'{s["speaker"]}: {s["text"]}' for s in chunk)
    ps = summarize(chunk_text)
    part_summaries.append(ps)
    print(f"Part {i}: {ps}")

summary_chunked = " ".join(part_summaries)
print("\nFULL SUMMARY:\n", summary_chunked)

Part 1: Student made an English poster about their environment. The project was carried out mostly outdoor. The last phase was conducted indoor.
Part 2: The student had a field trip to the countryside where they studied insects and plants. The project was successful and they received compliments from their professor.
Part 3: After the project, Student learned how to write contents and decorate posters from his science teacher, Mr. John. Most of his partners were friendly and responsible and he liked Michael, the leader of his team.
Part 4: After the project, the student knows how to work in a team and cooperate well with his teammates.

FULL SUMMARY:
 Student made an English poster about their environment. The project was carried out mostly outdoor. The last phase was conducted indoor. The student had a field trip to the countryside where they studied insects and plants. The project was successful and they received compliments from their professor. After the project, Student learned ho

In [7]:
import re, json
from google.colab import files

def neutral(t):
    for pat, rep in [(r"\bhis\b", "their"), (r"\bHis\b", "Their"),
                     (r"\bhe\b", "they"),   (r"\bHe\b", "They"),
                     (r"\bhim\b", "them")]:
        t = re.sub(pat, rep, t)
    return t

final = {
    "meeting_id": meeting["meeting_id"],
    "summary": neutral(summary_chunked),
    "summary_single_pass": summary_single,
    "speakers": {
        sp: {"segments": r["turns"],
             "talk_seconds": round(r["seconds"], 1),
             "entities": sorted(r["entities"]),
             "lines": r["lines"]}
        for sp, r in report.items()
    },
    "segments": meeting["segments"],
}

with open("report_sample1.json", "w", encoding="utf-8") as f:
    json.dump(final, f, indent=2, ensure_ascii=False)

print(final["summary"])
files.download("report_sample1.json")

Student made an English poster about their environment. The project was carried out mostly outdoor. The last phase was conducted indoor. The student had a field trip to the countryside where they studied insects and plants. The project was successful and they received compliments from their professor. After the project, Student learned how to write contents and decorate posters from their science teacher, Mr. John. Most of their partners were friendly and responsible and they liked Michael, the leader of their team. After the project, the student knows how to work in a team and cooperate well with their teammates.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>